In [11]:
import os
import json
import requests
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()

API_KEY = os.getenv("RM_Deepseek_R1_PaidKey")

BASE_URL = "https://api.deepseek.com/v1/chat/completions"
MODEL = "deepseek-reasoner"

print("API Key Loaded:", API_KEY[:10] + "...")

API Key Loaded: sk-0307ed0...


In [12]:
input_file = "safari_cleaned.json"

with open(input_file, "r", encoding="utf-8") as f:
    safari_data = json.load(f)

print("Total records:", len(safari_data))

# Choose your dynamic range:
FROM = 11
UPTO = 21

data_chunk = safari_data[FROM:UPTO]
print(f"Items selected for enrichment: {len(data_chunk)}")

Total records: 1101
Items selected for enrichment: 10


In [13]:
def build_prompt(item):
    return f"""
You are an expert Sri Lankan travel data enrichment AI.

Given this safari listing:

{json.dumps(item, indent=2)}

Extract and infer additional structured attributes. 
Return ONLY valid JSON with the following structure:

{{
  "title": "...",
  "image_links": [...],
  "rating": "...",
  "total_reviews": "...",
  "recommendation": "...",
  "type": "...",
  "travel_duration": "...",
  "description": "...",
  "starting_price": "...",
  "extracted_features": {{
      "park_names": [],
      "primary_wildlife_focus": "",
      "safari_type_detail": "",
      "suitability": "",
      "duration_category": "",
      "price_category": "",
      "key_attractions": [],
      "flexibility_notes": "",
      "district": "",
      "latitude": "",
      "longitude": ""
  }}
}}

Rules:
- ALWAYS return valid JSON.
- If uncertain, guess based on Sri Lankan geography.
- For district, latitude, and longitude → infer from the parks mentioned.
"""


In [14]:
def call_deepseek(prompt):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": MODEL,
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    response = requests.post(BASE_URL, headers=headers, json=payload)

    if response.status_code != 200:
        print("Error:", response.text)
        return None

    result = response.json()
    output = result["choices"][0]["message"]["content"]

    # Ensure JSON is parseable
    try:
        return json.loads(output)
    except json.JSONDecodeError:
        print("JSON parse error. Raw output:\n", output)
        return None


In [15]:
enriched_results = []

for item in tqdm(data_chunk, desc="Enriching safaris"):
    prompt = build_prompt(item)
    enriched = call_deepseek(prompt)
    
    if enriched:
        enriched_results.append(enriched)
    else:
        enriched_results.append(item)  # fallback

print("Completed!")


Enriching safaris:  80%|████████  | 8/10 [07:51<01:53, 56.77s/it]

JSON parse error. Raw output:
 ```json
{
  "title": "Ella to Mirissa Shuttle & Safari Drive (Breakfast Included)",
  "image_links": [
    "https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2f/fa/69/36/caption.jpg?w=500&h=400&s=1"
  ],
  "rating": "4.6",
  "total_reviews": "21",
  "recommendation": "Recommended by 90% of travelers",
  "type": "Day Trips",
  "travel_duration": "6+ hours",
  "description": "Enjoy a seamless door to door transfer between Ella to Mirissa via Udawalawe national park. Udawalawe National Park is renowned…",
  "starting_price": "40",
  "extracted_features": {
    "park_names": ["Udawalawe National Park"],
    "primary_wildlife_focus": "Elephants",
    "safari_type_detail": "Transfer Safari with Wildlife Viewing",
    "suitability": "Most travelers, Families, Wildlife enthusiasts",
    "duration_category": "Full Day",
    "price_category": "Mid-range",
    "key_attractions": ["Elephant sightings", "Wildlife photography", "Scenic transfer", "Breakfast inclu

Enriching safaris: 100%|██████████| 10/10 [10:20<00:00, 62.06s/it]

Completed!


In [6]:
output_file = "safaris_.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(enriched_results, f, indent=2, ensure_ascii=False)

print("Saved enriched data to:", output_file)


Saved enriched data to: safaris_.json
